# Chapter 06: Parsing, Chunking, and Retrieval Units

Offline walkthrough of exact source spans and three segmentation schemes.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


## Load the Part II fixture

In [ ]:
import json
from document_identity import Collection
from evidence_units import build_units, SCHEMES, parse_blocks, evidence_success

collection = Collection.load(ROOT / "examples" / "fixtures" / "part2-v1.json")
needs = json.load(open(ROOT / "examples" / "fixtures" / "part2-development.json"))["queries"]
print("schemes:", SCHEMES, "development needs:", len(needs))

## Exact spans reconstruct the parent text

In [ ]:
text = next(v["text"] for v in collection.fixture["versions"] if v["document_id"] == "maintenance")
blocks = parse_blocks(text)
for block in blocks:
    print(block["kind"], block["span"], repr(block["text"][:32]))
a0, b0 = blocks[0]["span"]
assert text[a0:b0] == blocks[0]["text"]
assert all(text[b["span"][0]:b["span"][1]] == b["text"] for b in blocks)

## Scoped evidence needs a heading plus an instruction

In [ ]:
need = next(q for q in needs if q["query_id"] == "s-d01")
for anchor in need["bundles"][0]:
    print(anchor["span"], repr(anchor["quote"]))
    assert anchor["span"][0] == 0 or anchor["span"][0] == 73
blocks_units = build_units(collection, "part2", "blocks")
print("blocks:", evidence_success(need, blocks_units))
assert evidence_success(need, blocks_units)["bundle_covered"] is True
assert evidence_success(need, blocks_units)["single_unit_sufficient"] is False

## Three deterministic segmentation policies

In [ ]:
for scheme in SCHEMES:
    units = build_units(collection, "part2", scheme)
    result = evidence_success(need, units)
    print(scheme, "units=", len(units), "single=", result["single_unit_sufficient"], "covered=", result["bundle_covered"])
    assert len(units) > 0
assert build_units(collection, "part2", "blocks") != build_units(collection, "part2", "sections")

## Unit sizes differ even at the same unit count budget

In [ ]:
for scheme in SCHEMES:
    units = build_units(collection, "part2", scheme)
    mean_chars = sum(len(u["text"]) for u in units) / len(units)
    print(scheme, "mean unit chars=", round(mean_chars, 1))
assert len(build_units(collection, "part2", "sections")) < len(build_units(collection, "part2", "blocks"))

## Try it yourself
Change `s-d01` to `s-d04` or `s-d06`, or inspect `parse_blocks` on another version.

In [ ]:
for query_id in ["s-d04", "s-d06", "s-d08"]:
    q = next(x for x in needs if x["query_id"] == query_id)
    print(query_id, [(u["passage_id"], ) for u in build_units(collection, "part2", "blocks")][:1])
    print("  bundles:", len(q["bundles"]))

## Observed versus not established
Observed: exact reconstruction, bundle coverage and unit counts for this fixture. Not established: relevance, consumer assembly, admission, use or helpfulness.